In [ ]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 



r_seed = 42

WORKDIR = 'C:/Users/thewr/git/LSTM_giroscopio_acelerometro.git/'
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/Code/DataPrep/"+data_hora

In [ ]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [ ]:
## get dataset

# select only sensor data from smart watches
data = pd.read_csv('normalizado.csv')

In [ ]:
data.shape


In [ ]:
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
#remove the null class
data=data.dropna()

# **Data Exploration**

In [ ]:
## the distribution of output classes
plt.bar(data["atividade"].value_counts().index, data["atividade"].value_counts().values)
plt.show()

In [ ]:
## x,y,z vaues for activity 'sit' 
data[(data['atividade'] == 0 )&(data['pessoa_x']==1600)][['acelerômetro_x','acelerômetro_Y','acelerômetro_Z']][:200].plot(title='0',subplots=True, figsize=(16, 12))

In [ ]:
## x,y,z vaues for activity 'stand' 
data[(data['atividade'] == 0 )&(data['pessoa_x']==1600)][['giroscópio_x','giroscópio_Y','giroscópio_Z']][:200].plot(title='0',subplots=True, figsize=(16, 12))

In [ ]:
## number of events per user

plt.bar(data["pessoa_x"].value_counts().index, data["pessoa_x"].value_counts().values)
plt.show()

In [ ]:
data = data[data["atividade"].isin([0,1,2,3])]

In [ ]:
data.shape

In [ ]:
data.head()

# **Models**

In [ ]:
#X = data[['acelerômetro_x','acelerômetro_Y','acelerômetro_Z','giroscópio_x','giroscópio_Y','giroscópio_Z']]
#y = data[['atividade']]





In [ ]:
#X.head()

In [ ]:
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=r_seed, shuffle=False)

In [ ]:
data = data.reset_index()

In [ ]:
#X = data.drop("target_column", axis=1)  # Replace "target_column" with the actual name
y = data["atividade"]

# Calculate indices for train and test sets, preserving class proportions
train_test_indices = []
val_indices = []
for class_value in y.unique():
   class_data = data[y == class_value]
   train_test_size = int(0.7 * len(class_data))
   train_test_indices.extend(class_data.index[:train_test_size])
   val_indices.extend(class_data.index[train_test_size:])

# Split the dataset without shuffling
print(len(train_test_indices))
print(len(val_indices))
X_train1 = data.iloc[train_test_indices]
X_test1 = data.iloc[val_indices]


# Print shapes to verify proportions
print("Train set:", X_train1.shape)
print("Test set:", X_test1.shape)

In [ ]:
#y_train = X_train[['atividade']]
X_train = X_train1[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z','atividade']]
#y_test = X_test[['atividade']]
X_test = X_test1[['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z','atividade']]

In [ ]:
def cria_IN_OUT(data, janela, features):
    OUT_indices = np.arange(janela, len(data), janela)
    #print("imprimindo out_indices")
    #print(OUT_indices)
    OUT = data[OUT_indices]
    #print("imprimindo out")
    #print(OUT)
    IN_indices = [range(i-janela, i) for i in OUT_indices] 
    #print("imprimindo IN_INDICES")
    #print(IN_indices)  
    #print("imprimindo IN_indices flattened") 
    #print(np.array(IN_indices).flatten())
    IN = data[np.array(IN_indices).flatten()]
    #print("imprimindo IN")
    #print(IN)      
    IN = np.reshape(IN, (len(OUT), janela, features))
    IN_final = IN[:, :, :]
    return IN_final

In [ ]:
featuresX = ['acelerômetro_x', 'acelerômetro_Y', 'acelerômetro_Z', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z','atividade']
trainX = np.empty((0, 200, len(featuresX)), dtype=float)
testX = np.empty((0, 200, len(featuresX)), dtype=float)
trainY = np.empty((0, 200, 1), dtype=float)
testY = np.empty((0, 200, 1), dtype=float)

arraytrain = X_train.values
print("imprimindo araytrain")
print(len(arraytrain))
arrayteste = X_test.values
print("imprimindo arraytest")
print(len(arrayteste))
transform = cria_IN_OUT(arraytrain, 200, 7)
transform2 = cria_IN_OUT(arrayteste, 200, 7)

trainX = np.concatenate([trainX, transform])
testX = np.concatenate([testX, transform2])

print(trainX.shape)
print(testX.shape)


In [ ]:
print(trainX)

In [ ]:
ind = np.arange(trainX.shape[0])

In [ ]:
print(len(ind))

In [ ]:
print(ind)

In [ ]:
np.random.shuffle(ind)

In [ ]:
trainX = trainX[ind,:,:]

In [ ]:
print(trainX)

In [ ]:
#optimizer = optimizers.Adam(clipvalue=1.0)

In [ ]:
trainY = trainX[:, :, -1]

In [ ]:
print(trainY)

In [ ]:
trainY.shape

In [ ]:
from scipy.stats import mode

In [ ]:
trainY_mode = mode(trainY,axis=1)[0]

In [ ]:
print(len(trainY_mode))

In [ ]:
print(trainY_mode)

In [ ]:
trainY_mode_categorical = to_categorical(trainY_mode)

In [ ]:
print(trainY_mode_categorical)

In [ ]:
trainX = trainX[:,:,:6]

In [ ]:
trainX.shape

In [ ]:
print(trainX)

In [ ]:
ind_test = np.arange(testX.shape[0])

In [ ]:
print(len(ind_test))

In [ ]:
print(ind_test)

In [ ]:
np.random.shuffle(ind_test)

In [ ]:
testX = testX[ind_test,:,:]

In [ ]:
print(testX)

In [ ]:
#optimizer = optimizers.Adam(clipvalue=1.0)

In [ ]:
testY = testX[:, :, -1]

In [ ]:
print(testY)

In [ ]:
testY.shape

In [ ]:
from scipy.stats import mode

In [ ]:
testY_mode = mode(testY,axis=1)[0]

In [ ]:
print(len(testY_mode))

In [ ]:
print(testY_mode)

In [ ]:
testY_mode_categorical = to_categorical(testY_mode)

In [ ]:
print(testY_mode_categorical)

In [ ]:
testX = testX[:,:,:6]

In [ ]:
testX.shape

In [ ]:
print(testX)

In [ ]:
def cria_LSTM(hp):
  model = tf.keras.Sequential()  
  model.add(tf.keras.layers.LSTM(hp.Int("input_units",min_value=32,max_value=256,step=32), input_shape=(200,6), return_sequences=False))
  model.add(tf.keras.layers.Dense(units=200, activation='relu'))
  #model.add(tf.keras.layers.BatchNormalization())
  model.add(tf.keras.layers.Dropout(rate=hp.Float(
                'dropout_1',
                min_value=0.4,
                max_value=0.5,
                default=0.4,
                step=0.05,
            )))
  ''' model.add(tf.keras.layers.Dropout(rate=hp.Float(
                'dropout_2',
                min_value=0.0,
                max_value=0.5,
                default=0.25,
                step=0.05,
            ))) '''
  model.add(tf.keras.layers.Dense(units=128, activation='relu'))
  model.add(tf.keras.layers.Dropout(rate=hp.Float(
                'dropout_2',
                min_value=0.4,
                max_value=0.5,
                default=0.4,
                step=0.05,
  )))  
  #model.add(tf.keras.layers.Dense(units=32, activation=activation[0]))
  model.add(tf.keras.layers.Dense(units=4, activation='softmax'))
  model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(hp.Float(
                    'learning_rate',
                    min_value=1e-4,
                    max_value=1e-2,
                    sampling='LOG',
                    default=1e-3
                )), metrics=['accuracy'])
  return model


In [ ]:
tuner = RandomSearch(
    cria_LSTM,
    objective = 'val_accuracy', 
    seed=42,
    max_trials = 5,        
    executions_per_trial = 1,    
    directory = LOG_DIR,
    project_name="lstm_acel_giro_multiclass")

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    min_delta = 0,
    verbose=1,
    restore_best_weights=True)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    verbose=1)

tensorboard = tf.keras.callbacks.TensorBoard(LOG_DIR)

my_callbacks = [early_stopping,reduce_lr,tensorboard]  

''' from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight('balanced',
                                                 np.unique(tweets_for_model_training_target.sentimento),
                                                 tweets_for_model_training_target.sentimento)
class_weights = {i : class_weights[i] for i in range(0,3)} '''


In [ ]:
tuner.search(x=trainX,
            y=trainY_mode_categorical,
            epochs=50,
            validation_data=(testX,testY_mode_categorical), callbacks=my_callbacks,batch_size=60)

In [ ]:
tuner.results_summary()

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
best_model.summary()

In [ ]:
loss, accuracy = best_model.evaluate(testX, testY_mode_categorical)

In [ ]:
predictions = best_model.predict(testX)

In [ ]:
print(predictions)

In [ ]:
print(testX)

In [ ]:
print(predictions)

In [ ]:
classes_x=np.argmax(predictions,axis=1)

In [ ]:
print(classes_x)

In [ ]:
print(len(classes_x))

In [ ]:
#classes_x_mode = mode(classes_x,axis=1)[0]

In [ ]:
#print(len(classes_x_mode))

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report
cm = confusion_matrix(testY_mode, classes_x)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

In [ ]:
disp.plot()

In [ ]:
print(classification_report(testY_mode, classes_x))

In [ ]:
x = np.arange(84).reshape((3,4,7))

In [ ]:
x.shape

In [ ]:
print(x)

In [ ]:
ind = np.arange(x.shape[0])

In [ ]:
print(ind)

In [ ]:
np.random.shuffle(ind)

In [ ]:
#permutation = [2,0,1]

In [ ]:
x_permuted = x[ind,:,:]

In [ ]:
print(x_permuted)

In [ ]:
#optimizer = optimizers.Adam(clipvalue=1.0)

In [ ]:
y= x_permuted[:, :, -1]

In [ ]:
print(y)

In [ ]:
y = y.reshape(-1)

In [ ]:
print(y)